# Generate And Visualize Results

This notebook runs the configured experiments one by one through the project CLI, normalizes the CLI stdout paths relative to the detected project root, and displays the resulting PNG files inline.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root(Path.cwd())
RESULTS_ROOT = PROJECT_ROOT / "results"
SRC_ROOT = PROJECT_ROOT / "src"

if shutil.which("dadmm-localization") is not None:
    CLI_PREFIX = ["dadmm-localization"]
else:
    CLI_PREFIX = [sys.executable, "-m", "dadmm_localization.cli"]

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RESULTS_ROOT = {RESULTS_ROOT}")
print(f"SRC_ROOT = {SRC_ROOT}")
print(f"CLI_PREFIX = {' '.join(CLI_PREFIX)}")

In [ ]:
EXPERIMENTS = [
    {
        "name": "Experiment 0",
        "config": PROJECT_ROOT / "configs" / "exp0_dadmm_quick_check_eight.yaml",
    },
    {
        "name": "Experiment 1",
        "config": PROJECT_ROOT / "configs" / "exp1_random_convergence_eight.yaml",
    },
    {
        "name": "Experiment 2",
        "config": PROJECT_ROOT / "configs" / "exp2_sigma_sweep_eight.yaml",
    },
    {
        "name": "Experiment 3",
        "config": PROJECT_ROOT / "configs" / "exp3_penalty_sweep_eight.yaml",
    },
]

pd.DataFrame(
    {
        "name": [item["name"] for item in EXPERIMENTS],
        "config": [str(item["config"].relative_to(PROJECT_ROOT)) for item in EXPERIMENTS],
    }
)

In [ ]:
def run_cli(args: list[str], cwd: Path) -> subprocess.CompletedProcess[str]:
    command = CLI_PREFIX + args
    display(Markdown(f"CLI: `{' '.join(command)}`"))
    try:
        completed = subprocess.run(
            command,
            cwd=cwd,
            check=True,
            text=True,
            capture_output=True,
            env={**os.environ, "PYTHONPATH": str(SRC_ROOT)},
        )
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            display(Markdown("**stdout**"))
            print(exc.stdout)
        if exc.stderr:
            display(Markdown("**stderr**"))
            print(exc.stderr)
        raise
    if completed.stdout.strip():
        display(Markdown("**stdout**"))
        print(completed.stdout)
    if completed.stderr.strip():
        display(Markdown("**stderr**"))
        print(completed.stderr)
    return completed


def normalize_cli_path(path_text: str, base_dir: Path) -> Path:
    candidate = Path(path_text.strip()).expanduser()
    if candidate.is_absolute():
        return candidate.resolve()
    return (base_dir / candidate).resolve()


def parse_run_dir(stdout: str, base_dir: Path) -> Path:
    lines = [line.strip() for line in stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError("CLI run command did not print a run directory.")
    return normalize_cli_path(lines[-1], base_dir)


def parse_figure_paths(stdout: str, base_dir: Path) -> list[Path]:
    lines = [line.strip() for line in stdout.splitlines() if line.strip()]
    return [normalize_cli_path(line, base_dir) for line in lines]


def display_generated_figures(image_paths: list[Path]) -> None:
    if not image_paths:
        display(Markdown("No figures were reported by the visualizer."))
        return
    for image_path in image_paths:
        display(Markdown(f"**{image_path.name}**  `{image_path}`"))
        display(Image(filename=str(image_path)))


def run_and_visualize_experiment(name: str, config_path: Path) -> dict[str, object]:
    if not config_path.exists():
        raise FileNotFoundError(f"Config does not exist: {config_path}")

    display(Markdown(f"## {name}"))
    display(Markdown(f"Config: `{config_path.relative_to(PROJECT_ROOT)}`"))
    display(Markdown(f"Working directory: `{PROJECT_ROOT}`"))

    run_result = run_cli(["run", "--config", str(config_path.relative_to(PROJECT_ROOT))], PROJECT_ROOT)
    run_dir = parse_run_dir(run_result.stdout, PROJECT_ROOT)
    display(Markdown(f"Run directory: `{run_dir}`"))

    visualize_result = run_cli(["visualize", "--run-dir", str(run_dir)], PROJECT_ROOT)
    image_paths = parse_figure_paths(visualize_result.stdout, PROJECT_ROOT)
    display_generated_figures(image_paths)

    return {
        "name": name,
        "config": str(config_path.relative_to(PROJECT_ROOT)),
        "run_dir": str(run_dir),
        "num_figures": len(image_paths),
        "figures": [str(path) for path in image_paths],
    }

In [ ]:
run_manifest = []

for experiment in EXPERIMENTS:
    run_manifest.append(
        run_and_visualize_experiment(
            name=experiment["name"],
            config_path=experiment["config"],
        )
    )

summary_df = pd.DataFrame(run_manifest)
summary_df